# Explore the genome with a linked brush

This notebook makes an interactive genome visualization with [GenomeSpy for Python](https://genomespy.app/genome-spy-python/).

Think of the top track as a **map of the whole genome**. Drag a box across it to choose a region. The three tracks underneath act like magnifying glasses: they all zoom to the region you selected.

You do not need to know GenomeSpy yet. Run each cell from top to bottom using the play button on its left.

## 1. Install GenomeSpy

**Local JupyterLab:** before opening this notebook, create one environment for both JupyterLab and GenomeSpy. Run these commands in a terminal using Python 3.11 or newer:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install jupyterlab "genome-spy-python[arrow]" pandas numpy
python -m jupyterlab
```

On Windows, use `.venv\Scripts\Activate.ps1` to activate in PowerShell. Select this environment's Python 3 kernel. Skip the installation cell below locally.

**Google Colab:** run the next cell to install the package. The `arrow` extra provides efficient table transport. If GenomeSpy was already imported, restart the runtime after installation.

In [ ]:
# Colab needs installation inside its temporary runtime; local setup is above.
import importlib.util

if importlib.util.find_spec("google") and importlib.util.find_spec("google.colab"):
    get_ipython().run_line_magic(
        "pip", 'install "genome-spy-python[arrow]" pandas numpy'
    )

## 2. Let Colab show interactive widgets

GenomeSpy charts are interactive widgets. The next cell switches on Colab's widget support. Outside Colab, the small `try`/`except` simply does nothing.

In [ ]:
try:
    # This import exists in Google Colab.
    from google.colab import output

    # Give custom widgets permission to draw inside the notebook.
    output.enable_custom_widget_manager()
except ImportError:
    # We are probably in Jupyter or VS Code, where this switch is not needed.
    pass

## 3. Load and prepare the example data

The package includes a small table of HapMap variants. Each row describes one variant, including its chromosome (`CHR`), position (`BP`), p-value (`P`), effect size, and Z-score.

GenomeSpy understands chromosome names such as `chr5`. We create that column and also turn tiny p-values into easier-to-plot `-log10(p)` values. This preparation happens in Python before the table is sent to GenomeSpy.

In [ ]:
import numpy as np

import genome_spy as gs

# Check the imported copy before building the interactive chart.
if not hasattr(gs, "selection_interval"):
    raise RuntimeError(
        f"GenomeSpy was imported from {gs.__file__}, but its selection API is missing. "
        "Restart the kernel/runtime after installation and rerun the notebook. "
        "If this persists, check that the selected kernel uses the installed package "
        "and that no local genome_spy.py file or genome_spy folder shadows it."
    )

from genome_spy.datasets import load_dataset

# Load the table that is packaged inside genome-spy-python.
variants = load_dataset("hapmap_gwas", as_format="dataframe")

# A logarithm cannot use zero, so keep only valid positive p-values.
variants = variants.loc[variants["P"] > 0].copy()

# Turn chromosome numbers into names that the hg18 genome assembly knows.
variants["chrom"] = np.where(
    variants["CHR"] == 23,
    "chrX",
    "chr" + variants["CHR"].astype(str),
)

# Small p-values become tall points. For example, 0.00001 becomes 5.
variants["neglog"] = -np.log10(variants["P"])
y_domain = [0, float(np.ceil(variants["neglog"].max()))]

print(f"Loaded {len(variants):,} variants")
variants[["SNP", "chrom", "BP", "P", "EFFECTSIZE", "ZSCORE"]].head()

## 4. Make a shared genomic selection

A **brush** is the blue rectangle you draw with the mouse. Its only job is to remember the genomic interval you selected.

The starting interval covers chromosome 5, so the detail tracks show something useful before you touch the chart.

In [ ]:
# This is the region shown when the chart first opens.
initial_region = [
    {"chrom": "chr5", "pos": 0},
    {"chrom": "chr5", "pos": 180_857_866},
]

# The empty parameter is the shared box where the selected interval is stored.
brush = gs.param("brush")

# This interval selection listens for horizontal dragging on the top track.
brush_update = gs.selection_interval(
    "brush",
    encodings=["x"],
    mark=gs.BrushConfig(
        clip=False,
        fill="#4c78a8",
        fillOpacity=0.20,
        stroke="#315f8c",
        strokeWidth=1.2,
        measure="outside",
    ),
    push="outer",
    persist=False,
)

## 5. Build the whole-genome overview

The `Locus` encoding combines a chromosome column and a position column into one continuous genome-wide axis. That is why points from chromosome 1 through chromosome X can share one x-axis.

We turn zooming off only for this overview so it always remains our complete genome map.

In [ ]:
overview_track = (
    gs.Chart()
    # Draw one small dot for every variant.
    .mark_point(filled=True, size=13, opacity=0.68, color="#7f8c8d")
    .encode(
        # Locus tells GenomeSpy: combine chromosome + base-pair position.
        x=(
            gs.Locus("chrom", "BP")
            .scale(assembly="hg18", zoom=False)
            .axis(title=None, chromTicks=True, chromLabels=True)
        ),
        y=(
            gs.Y("neglog:Q")
            .scale(domain=y_domain)
            .axis(title="Overview", grid=False, labels=False, ticks=False)
        ),
        tooltip=["SNP:N", "chrom:N", "BP:Q", "P:Q"],
    )
    .properties(height=105)
    # Connect mouse dragging on this track to our brush.
    .add_params(brush_update)
)

# Leave a little space above the track for the brush's base-pair measurement.
overview = (
    gs.vconcat(overview_track)
    .properties(padding=gs.Paddings(top=24))
    .resolve_scale(x="excluded")
)

## 6. Build three detail tracks

Each detail track draws a different measurement, but all three ask the same brush for their x-axis range. Imagine three magnifying glasses tied to the same handle: moving the handle moves all three together.

In [ ]:
# Reuse this domain in every detail track.
selected_region = gs.SelectionDomainRef(
    param=brush.name,
    initial=initial_region,
)

association_track = (
    gs.Chart()
    .mark_point(filled=True, size=24, opacity=0.78, color="#4c78a8")
    .encode(
        x=gs.Locus("chrom", "BP").scale(domain=selected_region).axis(None),
        y=gs.Y("neglog:Q").scale(domain=y_domain).title("−log10 p"),
        tooltip=["SNP:N", "GENE:N", "P:Q"],
    )
    .properties(height=95)
)

effect_track = (
    gs.Chart()
    .mark_point(filled=True, size=24, opacity=0.78, color="#f58518")
    .encode(
        x=gs.Locus("chrom", "BP").scale(domain=selected_region).axis(None),
        y=gs.Y("EFFECTSIZE:Q").scale(domain=[-3, 3]).title("Effect size"),
        tooltip=["SNP:N", "GENE:N", "EFFECTSIZE:Q"],
    )
    .properties(height=95)
)

zscore_track = (
    gs.Chart()
    .mark_point(filled=True, size=24, opacity=0.78, color="#54a24b")
    .encode(
        x=gs.Locus("chrom", "BP").scale(domain=selected_region),
        y=gs.Y("ZSCORE:Q").scale(domain=[0, 7]).title("Z-score"),
        tooltip=["SNP:N", "GENE:N", "ZSCORE:Q"],
    )
    .properties(height=95)
)

## 7. Stack the tracks and try the chart

`vconcat` stacks charts vertically. The whole composition receives the same data, genome assembly, and brush parameter.

After the chart appears:

1. **Drag across the gray overview** to select a region.
2. Watch all three detail tracks jump to that region.
3. Hover over a point to see its values.
4. Scroll over a detail track to zoom, or drag it to pan.

In [ ]:
chart = (
    gs.vconcat(overview, association_track, effect_track, zscore_track)
    .properties(
        data=variants,
        assembly="hg18",
        title="Brush-linked HapMap association tracks",
        description=(
            "A whole-genome selection controls three synchronized detail tracks."
        ),
        spacing=8,
    )
    # Make the shared brush available to every child track.
    .add_params(brush)
    # The overview keeps the whole genome; the details use the selected region.
    .resolve_scale(x="independent", y="independent")
    .resolve_axis(x="independent", y="independent")
)

# A chart shown as the last line of a cell renders automatically.
chart

## What did GenomeSpy do?

Python prepared the table and described the visualization. GenomeSpy then did the interactive work in your browser:

- the **locus scale** placed chromosomes and base-pair positions on one genomic axis;
- the **interval selection** remembered the region you brushed;
- the **selection-driven domains** kept all detail tracks synchronized; and
- the browser updated the visualization immediately without rerunning Python.

Next, try changing a mark color, track height, or the `initial_region`, and rerun the affected cells.

> **Data note:** The packaged table is a subset of HapMap variants adapted from the MIT-licensed [manhattanly](https://github.com/cran/manhattanly) and [Plotly](https://github.com/plotly/datasets/blob/master/Dash_Bio/Chromosomal/manhattan_data.csv) datasets. Coordinates and annotations are real, but the association statistics are simulated and are not biological findings.

In [ ]:
view = chart.widget()

# Display this object once in the notebook.
view